# Titanic — klasyfikacja (regresja logistyczna)
Notebook zgodny z wymaganiami z `zadanie_titanic.pdf`.


In [ ]:
import numpy as np
import pandas as pd
import glob
from pathlib import Path

def read_csv_smart(patterns):
    if isinstance(patterns, str):
        patterns = [patterns]
    candidates = []
    for p in patterns:
        candidates += [Path(p), Path("data")/p, Path("../data")/p, Path("/content")/p]
    for c in candidates:
        if "*" in str(c):
            m = glob.glob(str(c))
            if m:
                return pd.read_csv(m[0]), m[0]
        else:
            if c.exists():
                return pd.read_csv(c), str(c)
    for p in patterns:
        stem = Path(p).stem.split(" (")[0]
        m = glob.glob(f"**/{stem}*.csv", recursive=True)
        if m:
            return pd.read_csv(m[0]), m[0]
    raise FileNotFoundError

try:
    df, src = read_csv_smart(["train.csv", "titanic_train.csv", "titanic*.csv", "Titanic*.csv"])
    print("Wczytano CSV:", src, "| shape:", df.shape)
except FileNotFoundError:
    import seaborn as sns
    df = sns.load_dataset("titanic")
    print("Wczytano seaborn titanic | shape:", df.shape)

df.head()


## 1. Eksploracja danych (EDA)
1.1 Typy danych i braki
1.2 Liczba braków w każdej kolumnie
1.3 Proporcja osób, które przeżyły (ogółem oraz wg płci i klasy)
1.4 Wykres przeżywalności wg płci i klasy

In [ ]:
df.info()


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing.to_frame("braki")


In [ ]:
target = "survived" if "survived" in df.columns else ("Survived" if "Survived" in df.columns else None)
if target is None:
    raise ValueError("Brak kolumny docelowej survived/Survived.")

df[target].value_counts(normalize=True)


In [ ]:
surv_overall = df[target].mean()
surv_by_sex = df.groupby("sex")[target].mean() if "sex" in df.columns else None
surv_by_class = df.groupby("pclass")[target].mean() if "pclass" in df.columns else None
surv_by_sex_class = df.pivot_table(values=target, index="sex", columns="pclass", aggfunc="mean") if {"sex","pclass"}.issubset(df.columns) else None

surv_overall, surv_by_sex, surv_by_class, surv_by_sex_class


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.countplot(data=df, x=target)
plt.title("Rozkład zmiennej docelowej")
plt.xlabel("survived")
plt.ylabel("liczba")
plt.tight_layout()
plt.show()

if {"sex","pclass",target}.issubset(df.columns):
    plt.figure()
    sns.barplot(data=df, x="pclass", y=target, hue="sex")
    plt.title("Przeżywalność w zależności od płci i klasy")
    plt.xlabel("pclass")
    plt.ylabel("średnia przeżycia (0/1)")
    plt.tight_layout()
    plt.show()


## 2. Przygotowanie danych
Użyte kolumny: `survived, pclass, sex, age, sibsp, parch, fare, alone` (zgodnie z pdf).
Braki w `age` uzupełnione medianą. `sex` zakodowane one-hot.
Podział 80/20 z `stratify`.

In [ ]:
cols = [target, "pclass", "sex", "age", "sibsp", "parch", "fare", "alone"]
cols = [c for c in cols if c in df.columns]
d = df[cols].copy()

if "age" in d.columns:
    d["age"] = d["age"].fillna(d["age"].median())

if "sex" in d.columns:
    d = pd.get_dummies(d, columns=["sex"], drop_first=True)

X = d.drop(columns=[target])
y = d[target].astype(int)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape


## 3. Budowa modelu: regresja logistyczna (bazowa)
3.1 Trening
3.2 Współczynniki i 3 najważniejsze zmienne
3.3 Odds ratio i interpretacja dla `sex_male`

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=2000, solver="liblinear")
lr.fit(X_train, y_train)

coef = pd.Series(lr.coef_[0], index=X_train.columns).sort_values(key=lambda s: s.abs(), ascending=False)
intercept = float(lr.intercept_[0])
coef, intercept


**Interpretacja (3 najważniejsze zmienne):**

Poniżej wypisane są 3 cechy o największej wartości bezwzględnej współczynnika. Znak współczynnika mówi o tym, czy cecha zwiększa (`+`) czy zmniejsza (`-`) log-odds przeżycia.

In [ ]:
top3 = coef.head(3).to_frame("beta")
top3["odds_ratio"] = np.exp(top3["beta"])
top3


**Odds ratio:**

Odds ratio = exp(beta). Jeśli OR < 1, cecha zmniejsza szanse przeżycia; jeśli OR > 1, zwiększa. Dla `sex_male` zwykle OR < 1, czyli bycie mężczyzną zmniejsza szanse przeżycia (przy pozostałych cechach stałych).

In [ ]:
odds = np.exp(pd.Series(lr.coef_[0], index=X_train.columns)).sort_values()
odds.to_frame("odds_ratio")


## 4. Ewaluacja modelu
Metryki: Accuracy, Precision, Recall, F1, Specificity, ROC/AUC + macierz konfuzji.

In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score, precision_score,
    recall_score, f1_score, roc_curve, roc_auc_score
)

y_pred = lr.predict(X_test)
y_prob = lr.predict_proba(X_test)[:,1]

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) else 0.0

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "specificity": specificity,
    "auc": roc_auc_score(y_test, y_prob),
}
metrics


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.heatmap(cm, annot=True, fmt="d", cbar=False)
plt.title("Macierz konfuzji — regresja logistyczna")
plt.xlabel("Przewidywane")
plt.ylabel("Rzeczywiste")
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
fpr, tpr, thr = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
plt.plot([0,1],[0,1], linestyle="--", label="losowy")
plt.title("Krzywa ROC — regresja logistyczna")
plt.xlabel("FPR (1 - specificity)")
plt.ylabel("TPR (recall)")
plt.legend()
plt.tight_layout()
plt.show()


**Która metryka jest najważniejsza i dlaczego?**

W tym zadaniu wybór zależy od kosztu błędów. Jeśli ważniejsze jest nieprzeoczenie osób, które przeżyły (FN kosztowne), priorytetem będzie Recall. Jeśli ważniejsze jest, by przewidywane przeżycia były wiarygodne (FP kosztowne), priorytetem będzie Precision. Dobrą metryką kompromisową bywa F1, a do porównań niezależnych od progu — AUC.

## 5. Eksperymenty
5.1 Zmiana progu (0.3–0.7): Precision i Recall
5.2 `class_weight='balanced'` vs bazowy

In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
rows = []
for t in thresholds:
    pred_t = (y_prob >= t).astype(int)
    rows.append({
        "threshold": t,
        "precision": precision_score(y_test, pred_t, zero_division=0),
        "recall": recall_score(y_test, pred_t, zero_division=0),
        "f1": f1_score(y_test, pred_t, zero_division=0),
    })
pd.DataFrame(rows)


In [ ]:
lr_bal = LogisticRegression(max_iter=2000, solver="liblinear", class_weight="balanced")
lr_bal.fit(X_train, y_train)

y_pred_b = lr_bal.predict(X_test)
y_prob_b = lr_bal.predict_proba(X_test)[:,1]

cm_b = confusion_matrix(y_test, y_pred_b)
tn, fp, fn, tp = cm_b.ravel()
spec_b = tn / (tn + fp) if (tn + fp) else 0.0

{
    "accuracy": accuracy_score(y_test, y_pred_b),
    "precision": precision_score(y_test, y_pred_b, zero_division=0),
    "recall": recall_score(y_test, y_pred_b, zero_division=0),
    "f1": f1_score(y_test, y_pred_b, zero_division=0),
    "specificity": spec_b,
    "auc": roc_auc_score(y_test, y_prob_b),
}


## 6. Regularyzacja
6.1 Standaryzacja zmiennych numerycznych (fit tylko na train)
6.2 Wpływ C (L2): AUC i suma |beta|
6.3 Porównanie L1 vs L2 dla C=0.1

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = [c for c in ["age", "sibsp", "parch", "fare"] if c in X.columns]
scaler = StandardScaler()

X_train_s = X_train.copy()
X_test_s = X_test.copy()

if len(num_cols):
    X_train_s[num_cols] = scaler.fit_transform(X_train_s[num_cols])
    X_test_s[num_cols] = scaler.transform(X_test_s[num_cols])

X_train_s.shape, X_test_s.shape


In [ ]:
Cs = [0.001, 0.01, 0.1, 1, 10, 100]
rows = []
for C in Cs:
    m = LogisticRegression(max_iter=3000, solver="liblinear", penalty="l2", C=C)
    m.fit(X_train_s, y_train)
    prob = m.predict_proba(X_test_s)[:,1]
    auc = roc_auc_score(y_test, prob)
    sum_abs_beta = float(np.sum(np.abs(m.coef_[0])))
    rows.append({"C": C, "auc": auc, "sum_abs_beta": sum_abs_beta})
res_C = pd.DataFrame(rows)
res_C


In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(res_C["C"], res_C["auc"], marker="o", label="AUC")
plt.plot(res_C["C"], res_C["sum_abs_beta"], marker="o", label="suma |beta|")
plt.xscale("log")
plt.title("Wpływ parametru C (L2): AUC i suma |beta|")
plt.xlabel("C (skala log)")
plt.ylabel("wartość")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
m_l2 = LogisticRegression(max_iter=3000, solver="liblinear", penalty="l2", C=0.1)
m_l1 = LogisticRegression(max_iter=3000, solver="liblinear", penalty="l1", C=0.1)

m_l2.fit(X_train_s, y_train)
m_l1.fit(X_train_s, y_train)

coef_comp = pd.DataFrame({
    "beta_L2": m_l2.coef_[0],
    "beta_L1": m_l1.coef_[0],
}, index=X_train_s.columns)
coef_comp["odds_L2"] = np.exp(coef_comp["beta_L2"])
coef_comp["odds_L1"] = np.exp(coef_comp["beta_L1"])
coef_comp.sort_values(by="beta_L2", key=lambda s: s.abs(), ascending=False)
